# 1. StateGraph Basics

The foundational LangGraph building block: a typed state schema, nodes (plain
functions that take the state and return a partial update), and edges wiring
nodes together into a graph you compile and invoke.

Contrast with LangChain's LCEL chains (a linear pipe): a `StateGraph` is nodes +
edges over a shared, typed state any node can read/write — which is what makes
branching and cycles (later notebooks) possible.

**Prerequisites:** Ollama running locally with `llama3.2` pulled.

### Setup

This cell makes the project's shared `tools`/`models` packages importable
regardless of where Jupyter's working directory actually is (it's usually
this notebook's own folder, not the repo root), and loads `.env` plus any
cached secrets in `.env.local` (populated by `scripts/lib/env.sh` the first
time you've run `scripts/start_app.sh` / `scripts/start_infra.sh`).

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

load_dotenv(project_root / ".env")
load_dotenv(project_root / ".env.local", override=True)  # cached secrets, if resolve_env() has run at least once
print("Project root on sys.path:", project_root)

In [ ]:
from typing import TypedDict

from langgraph.graph import END, START, StateGraph

from models.chat_models.ollama_models import SupportedModel, get_chat_model


class FactJokeState(TypedDict):
    topic: str
    fact: str
    joke: str


llm = get_chat_model(SupportedModel.llama3_2)


def generate_fact(state: FactJokeState) -> dict:
    response = llm.invoke(f"Give one interesting, concise fact about {state['topic']}.")
    return {"fact": response.content}


def generate_joke(state: FactJokeState) -> dict:
    response = llm.invoke(f"Write one short joke based on this fact: {state['fact']}")
    return {"joke": response.content}


graph = StateGraph(FactJokeState)
graph.add_node("generate_fact", generate_fact)
graph.add_node("generate_joke", generate_joke)
graph.add_edge(START, "generate_fact")
graph.add_edge("generate_fact", "generate_joke")
graph.add_edge("generate_joke", END)
compiled = graph.compile()

In [ ]:
result = compiled.invoke({"topic": "octopuses", "fact": "", "joke": ""})
print("fact:", result["fact"])
print()
print("joke:", result["joke"])

## Visualize the graph

In [ ]:
print(compiled.get_graph().draw_mermaid())

## 🧪 Playground

**1. A third node** — add `generate_pun` after `generate_joke`, wired the same way.

In [ ]:
# TODO: add a generate_pun node + field, rewire graph.add_edge("generate_joke", "generate_pun") -> END


**2. A different topic** — rerun with something more obscure and see how specific the fact is.

In [ ]:
# TODO: compiled.invoke with a different topic


**3. Inspect intermediate state** — call `generate_fact` directly as a plain function (not through the graph) and print its return value.

In [ ]:
# TODO: generate_fact({'topic': 'volcanoes', 'fact': '', 'joke': ''})
